In [1]:
!pip install rdkit scikit-learn pandas numpy matplotlib seaborn

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 37.1/37.1 MB 37.2 MB/s eta 0:00:00


In [4]:
import pandas as pd
import numpy as np

# Load CSV — separator is semicolon
df = pd.read_csv('/content/Plasmodium falciparum.csv', sep=';', low_memory=False)

print(df.shape)
print(df.columns.tolist())

(61587, 48)
['Molecule ChEMBL ID', 'Molecule Name', 'Molecule Max Phase', 'Molecular Weight', '#RO5 Violations', 'AlogP', 'Compound Key', 'Smiles', 'Standard Type', 'Standard Relation', 'Standard Value', 'Standard Units', 'pChEMBL Value', 'Data Validity Comment', 'Comment', 'Uo Units', 'Ligand Efficiency BEI', 'Ligand Efficiency LE', 'Ligand Efficiency LLE', 'Ligand Efficiency SEI', 'Potential Duplicate', 'Assay ChEMBL ID', 'Assay Description', 'Assay Type', 'BAO Format ID', 'BAO Label', 'Assay Organism', 'Assay Tissue ChEMBL ID', 'Assay Tissue Name', 'Assay Cell Type', 'Assay Subcellular Fraction', 'Assay Parameters', 'Assay Variant Accession', 'Assay Variant Mutation', 'Target ChEMBL ID', 'Target Name', 'Target Organism', 'Target Type', 'Document ChEMBL ID', 'Source ID', 'Source Description', 'Document Journal', 'Document Year', 'Cell ChEMBL ID', 'Properties', 'Action Type', 'Standard Text Value', 'Value']


In [5]:
# Keep relevant columns
df = df[['Molecule ChEMBL ID', 'Smiles', 'Standard Type', 'Standard Value', 'Standard Units']]

# Keep only IC50
df = df[df['Standard Type'] == 'IC50']

# Rename columns
df.columns = ['chembl_id', 'smiles', 'standard_type', 'standard_value', 'units']

# Drop missing
df = df.dropna(subset=['smiles', 'standard_value'])

# Convert to numeric
df['standard_value'] = pd.to_numeric(df['standard_value'], errors='coerce')
df = df.dropna(subset=['standard_value'])

# Drop duplicates
df = df.drop_duplicates(subset='chembl_id')

print(df.shape)
print(df['standard_value'].describe())

(27950, 5)
count    2.795000e+04
mean     1.681087e+05
std      8.501170e+06
min      0.000000e+00
25%      4.933500e+01
50%      8.500000e+02
75%      7.943280e+03
max      9.549926e+08
Name: standard_value, dtype: float64


In [6]:
# IC50 < 1000 nM = Active
df['label'] = df['standard_value'].apply(lambda x: 1 if x < 1000 else 0)

print(df['label'].value_counts())
print(f"\nActive: {df['label'].sum()} ({df['label'].mean()*100:.1f}%)")
print(f"Inactive: {(df['label']==0).sum()} ({(1-df['label'].mean())*100:.1f}%)")

label
1    14435
0    13515
Name: count, dtype: int64

Active: 14435 (51.6%)
Inactive: 13515 (48.4%)


In [7]:
from rdkit import Chem
from rdkit.Chem import rdFingerprintGenerator

generator = rdFingerprintGenerator.GetMorganGenerator(radius=2, fpSize=2048)

def smiles_to_fp(smiles):
    mol = Chem.MolFromSmiles(smiles)
    if mol is None:
        return None
    return list(generator.GetFingerprint(mol))

df['fingerprint'] = df['smiles'].apply(smiles_to_fp)
df = df.dropna(subset=['fingerprint'])

print(df.shape)

(27950, 7)


In [8]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, roc_auc_score

X = np.array(df['fingerprint'].tolist())
y = df['label'].values

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# 50 trees to keep file size small
model = RandomForestClassifier(n_estimators=50, random_state=42, n_jobs=-1)
model.fit(X_train, y_train)

preds = model.predict(X_test)
print(classification_report(y_test, preds))
print("ROC-AUC:", roc_auc_score(y_test, model.predict_proba(X_test)[:,1]))

              precision    recall  f1-score   support

           0       0.84      0.87      0.86      2703
           1       0.88      0.85      0.86      2887

    accuracy                           0.86      5590
   macro avg       0.86      0.86      0.86      5590
weighted avg       0.86      0.86      0.86      5590

ROC-AUC: 0.9276268231900795


In [9]:
import pickle
import os

with open('rf_model.pkl', 'wb') as f:
    pickle.dump(model, f)

size = os.path.getsize('rf_model.pkl') / (1024*1024)
print(f"Model saved! File size: {size:.1f} MB")

Model saved! File size: 37.0 MB
